# Solutions — Utilities and Security (Hard 16)

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

df_transactions = spark.table("samples.bakehouse.sales_transactions")
df_customers    = spark.table("samples.bakehouse.sales_customers")


## Solution 1 — String Utilities: concat_ws, ltrim, rtrim, levenshtein

In [ ]:
result_1 = (
    df_customers
    .withColumn("full_name",    F.concat_ws(" ", F.col("firstName"), F.col("lastName")))
    .withColumn("email_trimmed", F.ltrim(F.rtrim(F.col("emailAddress"))))
    .withColumn("name_dist_from_a",
        F.levenshtein(F.col("firstName"), F.lit("Alice"))
    )
    .select("customerID", "full_name", "email_trimmed", "name_dist_from_a")
    .limit(100)
)


## Solution 2 — Hashing for Data Masking: md5, sha2, hash, crc32

In [ ]:
result_2 = (
    df_customers
    .withColumn("email_md5",    F.md5(F.col("emailAddress")))
    .withColumn("email_sha256", F.sha2(F.col("emailAddress"), 256))
    .withColumn("email_crc32",  F.crc32(F.col("emailAddress")))
    .select("customerID", "emailAddress", "email_md5", "email_sha256", "email_crc32")
    .limit(100)
)


## Solution 3 — Encoding: base64 and unbase64

In [ ]:
result_3 = (
    df_customers
    .withColumn("email_b64",   F.base64(F.col("emailAddress")))
    .withColumn("email_back",  F.unbase64(F.col("email_b64")).cast("string"))
    .select("customerID", "emailAddress", "email_b64", "email_back")
    .limit(100)
)


## Solution 4 — Row Metadata: monotonically_increasing_id and spark_partition_id

In [ ]:
result_4 = (
    df_transactions
    .withColumn("row_id",      F.monotonically_increasing_id())
    .withColumn("partition_id", F.spark_partition_id())
    .select("transactionID", "row_id", "partition_id")
    .limit(100)
)


## Solution 5 — Global Temp Views: createGlobalTempView

In [ ]:
df_transactions.createOrReplaceGlobalTempView("global_transactions")
result_5 = spark.sql(
    "SELECT paymentMethod, COUNT(*) AS cnt FROM global_temp.global_transactions GROUP BY paymentMethod"
)


## Solution 6 — crossJoin, exceptAll, intersectAll

In [ ]:
# Small samples for cross join to avoid explosion
products  = df_transactions.select("product").distinct().limit(3)
payments  = df_transactions.select("paymentMethod").distinct().limit(3)

cross     = products.crossJoin(payments)
all_prod  = df_transactions.select("product")
top_prod  = df_transactions.filter(F.col("totalPrice") > 100).select("product")
except_df = all_prod.exceptAll(top_prod)
inter_df  = all_prod.intersectAll(top_prod)

result_6 = cross.union(
    except_df.select(F.col("product"), F.lit("N/A").alias("paymentMethod"))
).union(
    inter_df.select(F.col("product"), F.lit("N/A").alias("paymentMethod"))
).limit(9)


## Solution 7 — mapInPandas

In [ ]:
import pandas as pd

def add_bucket(iterator):
    for pdf in iterator:
        pdf["price_bucket"] = pd.cut(
            pdf["totalPrice"],
            bins=[0, 10, 50, 100, float("inf")],
            labels=["low", "medium", "high", "premium"],
        ).astype(str)
        yield pdf

schema = T.StructType(df_transactions.schema.fields + [
    T.StructField("price_bucket", T.StringType(), True)
])

result_7 = df_transactions.mapInPandas(add_bucket, schema=schema)


## Solution 8 — Math Utilities and repartitionByRange

In [ ]:
result_8 = (
    df_transactions
    .withColumn("sqrt_price",    F.round(F.sqrt(F.col("totalPrice")), 4))
    .withColumn("sign_price",    F.signum(F.col("totalPrice") - F.avg("totalPrice").over(Window.orderBy(F.lit(1)))))
    .withColumn("nanvl_example", F.nanvl(F.col("totalPrice").cast("double"), F.lit(0.0)))
    .repartitionByRange(4, "totalPrice")
    .select("transactionID", "totalPrice", "sqrt_price", "sign_price", "nanvl_example")
)
